# Setup

In [ ]:
!pip install -qU langchain accelerate bitsandbytes transformers chromadb sentence-transformers langchain_huggingface pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.7/300.7 kB 9.4 MB/s eta 0:00:00


Przeanalizujmy poszczególne biblioteki:

`langchain` - to framework do budowania aplikacji wykorzystujących modele językowe. Pozwala na tworzenie łańcuchów operacji, które umożliwiają modelom językowym wykonywanie złożonych zadań.

`accelerate` - narzędzie opracowane przez Hugging Face, które umożliwia łatwiejsze trenowanie modeli na wielu procesorach GPU lub TPU.

`bitsandbytes` - biblioteka do kwantyzacji modeli, co pozwala zmniejszyć ich rozmiar i przyspieszyć działanie przy niewielkiej utracie jakości.

`transformers` - główna biblioteka Hugging Face zawierająca implementacje modeli transformerowych, takich jak BERT, GPT, T5 i wiele innych.

`chromadb` - wektorowa baza danych, która jest często używana do przechowywania i wyszukiwania podobieństw między dokumentami.

`sentence-transformers` - specjalistyczna biblioteka do tworzenia wektorowych reprezentacji zdań, przydatna przy wyszukiwaniu semantycznym.

`langchain_huggingface` - rozszerzenie langchain do integracji z modelami Hugging Face.

`pypdf` - narzędzie do pracy z dokumentami PDF, często używane do ekstrakcji tekstu.

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import transformers

from langchain.document_loaders import PyPDFLoader
from langchain.vectorstores import Chroma
from langchain.llms import HuggingFacePipeline
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from time import time
from langchain_huggingface import HuggingFaceEmbeddings

import warnings
warnings.filterwarnings('ignore')

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

Importy można podzielić na kilka kategorii:

Biblioteki podstawowe:
- `os` - zapewnia interfejs do funkcji systemu operacyjnego
- `torch` - framework do uczenia maszynowego PyTorch
- `time` - umożliwia mierzenie czasu wykonania operacji

Biblioteki związane z modelami językowymi:
- `transformers` - zawiera implementacje różnych modeli językowych
- `AutoModelForCausalLM` - automatycznie ładuje modele generatywne
- `AutoTokenizer` - ładuje tokenizery dopasowane do modeli
- `BitsAndBytesConfig` - umożliwia konfigurację kwantyzacji modeli (zmniejszenie rozmiaru modeli)

Biblioteki LangChain do przetwarzania dokumentów:
- `PyPDFLoader` - służy do wczytywania plików PDF
- `Chroma` - baza wektorowa do przechowywania osadzonych tekstów
- `HuggingFacePipeline` - adapter do modeli z biblioteki transformers
- `RecursiveCharacterTextSplitter` - dzieli tekst na mniejsze fragmenty
- `RetrievalQA` - buduje system pytań i odpowiedzi oparty na wyszukiwaniu
- `HuggingFaceEmbeddings` - konwertuje tekst na wektory osadzeń

In [ ]:
class CFG:
    model = 'speakleash/Bielik-11B-v2.2-Instruct'
    device = 'cuda'
    temperature = 0.1
    repetition_penalty = 1.1
    max_new_tokens = 1000
    topk = 200
    top1 = 1
    sentence = 'ipipan/silver-retriever-base-v1.1'
    rag_data = "/content/ai_act_pl.pdf"
    dtype = torch.bfloat16

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name= CFG.sentence, model_kwargs={"device": CFG.device})

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.35k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/907k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/556k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.30M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/144 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Funkcje

In [ ]:
def test_model(tokenizer, pipeline, prompt):

    time_1 = time()
    sequences = pipeline(
        prompt, do_sample=True,
        top_k=10, num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
    )
    time_2 = time()
    print(f"Test inference: {round(time_2-time_1, 3)} sec.")
    for seq in sequences:
        print(f"Result: {seq['generated_text']}")


Funkcja przyjmuje trzy parametry:

1. `tokenizer` - obiekt tokenizera, który zamienia tekst na numery (tokeny) zrozumiałe dla modelu. Tokenizer dzieli tekst na mniejsze jednostki, które są wejściem dla modelu językowego.

2. `pipeline` - gotowy potok przetwarzania, który zarządza całym procesem generowania tekstu przez model języka.

3. `prompt` - zapytanie tekstowe, na które model ma odpowiedzieć. Jest to tekst wejściowy inicjujący generowanie treści.

Wywoływany jest potok przetwarzania (`pipeline`) z kilkoma parametrami:
- `prompt` - przekazane zapytanie
- `do_sample=True` - włącza tryb próbkowania, co oznacza, że model będzie wybierał słowa z pewnym elementem losowości
- `top_k=10` - ogranicza wybór do 10 najbardziej prawdopodobnych tokenów przy każdym kroku generowania
- `num_return_sequences=1` - wskazuje, że model ma zwrócić tylko jedną odpowiedź
- `eos_token_id=tokenizer.eos_token_id` - określa, jaki token oznacza koniec sekwencji, dzięki czemu model wie, kiedy zakończyć generowanie

In [ ]:
def test_rag(qa, prompt):
    time_1 = time()
    result = qa.run(prompt)
    time_2 = time()
    print(f"Inference time: {round(time_2-time_1, 3)} sec.")
    print("\nResult: ", result)

Ta funkcja `test_rag` służy do testowania systemu RAG (Retrieval Augmented Generation) i mierzenia czasu potrzebnego na wygenerowanie odpowiedzi wspartej wyszukiwaniem informacji.

Funkcja przyjmuje dwa parametry:
1. `qa` - obiekt systemu pytań i odpowiedzi, który łączy wyszukiwanie informacji z bazy wiedzy z generowaniem odpowiedzi przez model językowy
2. `prompt` - zapytanie użytkownika, na które system ma odpowiedzieć

Działanie funkcji jest następujące:

Następnie uruchamiany jest system RAG poprzez wywołanie metody `run` na obiekcie `qa` z zapytaniem `prompt`. W tym momencie zachodzi złożony proces, który obejmuje:
- przekształcenie zapytania użytkownika na wektor
- przeszukanie bazy wektorowej w celu znalezienia najbardziej odpowiednich fragmentów tekstu
- połączenie tych fragmentów z zapytaniem użytkownika
- przekazanie rozszerzonego kontekstu do modelu językowego
- wygenerowanie odpowiedzi przez model językowy na podstawie dostarczonego kontekstu

# Model

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=False,
)

Ta linia kodu tworzy konfigurację kwantyzacji, która jest techniką optymalizacji modelu językowego, pozwalającą na znaczne zmniejszenie zużycia pamięci przy niewielkim spadku jakości.

Obiekt `quantization_config` wykorzystuje klasę `BitsAndBytesConfig` z biblioteki Transformers, która umożliwia zastosowanie kwantyzacji 4-bitowej - techniki opracowanej przez zespół Bits and Bytes.

Parametry konfiguracji mają następujące znaczenie:

`load_in_4bit=True` - włącza ładowanie modelu w formacie 4-bitowym zamiast standardowych 32 lub 16 bitów. Oznacza to, że każda waga modelu będzie zajmować tylko 4 bity pamięci zamiast 16 czy 32, co daje 4-8 krotną redukcję zapotrzebowania na pamięć. Jest to ogromna korzyść przy pracy z dużymi modelami, takimi jak Bielik-11B, który normalnie wymagałby kilkudziesięciu GB pamięci RAM.

`bnb_4bit_quant_type="nf4"` - określa typ kwantyzacji jako NF4 (Normalized Float 4-bit). Jest to specjalny format danych opracowany właśnie dla modeli językowych, który lepiej zachowuje dokładność w porównaniu do standardowej kwantyzacji liniowej. Format NF4 został zaprojektowany z myślą o wartościach wag występujących w sieciach neuronowych, co pozwala zminimalizować utratę jakości.

`bnb_4bit_compute_dtype="float16"` - mimo że wagi są przechowywane w formacie 4-bitowym, obliczenia będą wykonywane w formacie float16 (16-bitowym). Jest to kompromis między precyzją a efektywnością - wagi zajmują mało miejsca w pamięci, ale obliczenia zachowują wyższą precyzję.

`bnb_4bit_use_double_quant=False` - wyłącza podwójną kwantyzację, która byłaby dodatkowym poziomem kompresji. Podwójna kwantyzacja mogłaby jeszcze bardziej zmniejszyć zużycie pamięci, ale kosztem większej utraty jakości modelu.

Dzięki tej konfiguracji, model Bielik-11B, który normalnie wymagałby około 22 GB pamięci w formacie float16, może działać na urządzeniu z około 5-6 GB dostępnej pamięci GPU. Jest to kluczowe dla umożliwienia uruchamiania dużych modeli językowych na przeciętnym sprzęcie, bez konieczności posiadania najdroższych kart graficznych.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
                CFG.model , quantization_config = quantization_config,
                torch_dtype = CFG.dtype,)
tokenizer = AutoTokenizer.from_pretrained(CFG.model)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


config.json:   0%|          | 0.00/598 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now default to True since model is quantized.


model.safetensors.index.json:   0%|          | 0.00/37.3k [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/27.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/3.49k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

```python
model = AutoModelForCausalLM.from_pretrained(
                CFG.model, quantization_config = quantization_config,
                torch_dtype = CFG.dtype,)
```

Klasa `AutoModelForCausalLM` automatycznie wykrywa i ładuje odpowiedni typ modelu na podstawie nazwy (`CFG.model`, czyli 'speakleash/Bielik-11B-v2.2-Instruct').

Parametry funkcji `from_pretrained` to:
- `CFG.model` - identyfikator modelu, w tym przypadku polski model Bielik-11B
- `quantization_config` - wcześniej utworzony obiekt konfiguracji kwantyzacji 4-bitowej, który drastycznie zmniejsza zapotrzebowanie na pamięć
- `torch_dtype` - typ danych używany przez model, ustawiony na `torch.bfloat16` w konfiguracji, co daje dobry kompromis między dokładnością a zużyciem pamięci

Po załadowaniu modelu, kod inicjalizuje tokenizer:

```python
tokenizer = AutoTokenizer.from_pretrained(CFG.model)
```

Tokenizer jest narzędziem, które zamienia tekst na sekwencje liczb (tokenów) zrozumiałe dla modelu. Musi on dokładnie odpowiadać modelowi, dlatego ładowany jest z tego samego źródła.

Dwie ostatnie linie konfigurują szczegóły działania tokenizera:

```python
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
```

- `tokenizer.pad_token = tokenizer.eos_token` - ustawia token wypełnienia (używany gdy trzeba uzupełnić sekwencje do jednakowej długości) jako token końca sekwencji. Jest to standardowa praktyka dla modeli typu GPT, które często nie mają domyślnie zdefiniowanego tokenu wypełnienia.

- `tokenizer.padding_side = "right"` - określa, że wypełnienie ma być dodawane z prawej strony sekwencji. Jest to ważne, ponieważ dodawanie tokenów wypełnienia z lewej strony zmieniłoby kontekst dla modelu i mogłoby pogorszyć jakość generowania.

In [ ]:
text_generation_pipeline = transformers.pipeline(
    model=model,
    tokenizer=tokenizer,
    temperature= CFG.temperature,
    task="text-generation",
    repetition_penalty= CFG.repetition_penalty,
    return_full_text=True,
    max_new_tokens= CFG.max_new_tokens,
)

Device set to use cuda:0


In [ ]:
llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

Ta linia kodu tworzy interfejs między biblioteką LangChain a potokiem generowania tekstu z Transformers. Jest to kluczowy element integrujący model językowy z systemem wyszukiwania i odpowiadania na pytania (RAG).

`HuggingFacePipeline` to klasa z biblioteki LangChain, która działa jako adapter - pozwala używać potoku Transformers w ekosystemie LangChain. Umożliwia to bezproblemową integrację modelu językowego z innymi komponentami LangChain, takimi jak retrievery czy łańcuchy pytań i odpowiedzi.

Gdy tworzymy obiekt `llm` za pomocą `HuggingFacePipeline(pipeline=text_generation_pipeline)`, budujemy most między wcześniej skonfigurowanym potokiem tekstowym a architekturą LangChain. Ten obiekt `llm` może teraz być używany wszędzie, gdzie LangChain oczekuje modelu językowego.

Jest to ważne, ponieważ tworzy abstrakcję nad specyficznym modelem - LangChain nie musi wiedzieć, że używany jest akurat model Bielik-11B z kwantyzacją 4-bitową. Zamiast tego, widzi tylko ogólny interfejs modelu językowego, który potrafi przyjąć tekst i wygenerować odpowiedź.

Ta abstrakcja daje wiele korzyści:
- Pozwala na łatwą wymianę modeli bez zmiany reszty kodu
- Umożliwia korzystanie z narzędzi LangChain, takich jak łańcuchy czy agenty
- Zapewnia spójny interfejs niezależnie od wybranego modelu

Dzięki tej linii, nasz polski model Bielik staje się pełnoprawnym komponentem ekosystemu LangChain, gotowym do współpracy z bazami wektorowymi i innymi elementami systemu RAG.

# Dane

In [ ]:
loader = PyPDFLoader(CFG.rag_data)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
all_splits = text_splitter.split_documents(documents)

`loader = PyPDFLoader(CFG.rag_data)` - Tworzy obiekt ładujący dokumenty PDF. Parametr `CFG.rag_data` wskazuje na ścieżkę do pliku PDF (w tym przypadku "/content/ai_act_pl.pdf", który prawdopodobnie zawiera polski tekst dotyczący regulacji sztucznej inteligencji). Loader jest odpowiedzialny za odczytanie tekstu z dokumentu PDF.

`documents = loader.load()` - Wywołuje metodę wczytującą dokument. W wyniku otrzymujemy listę obiektów Document, gdzie każdy obiekt reprezentuje jedną stronę dokumentu PDF. Każdy taki obiekt zawiera tekst strony oraz metadane (np. numer strony).

`text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)` - Tworzy obiekt dzielący tekst na mniejsze fragmenty (chunki). Podział tekstu jest konieczny, ponieważ modele osadzeń mają ograniczenie co do długości tekstu, który mogą przetworzyć za jednym razem. Dodatkowo, mniejsze fragmenty umożliwiają bardziej precyzyjne wyszukiwanie.

Parametr `chunk_size=500` oznacza, że każdy fragment będzie miał około 500 znaków. Jest to stosunkowo mały rozmiar fragmentu, który pozwala na bardzo precyzyjne wyszukiwanie, choć kosztem większej liczby fragmentów.

Parametr `chunk_overlap=20` określa, ile znaków ma się nakładać między sąsiednimi fragmentami. Nakładanie się fragmentów jest ważne, aby kontekst nie został utracony na granicach podziału.  

`all_splits = text_splitter.split_documents(documents)` - Wykonuje właściwy podział dokumentów na mniejsze fragmenty. Metoda `split_documents` przetwarza wszystkie strony dokumentu i dzieli je na fragmenty zgodnie z określonymi parametrami. Wynikiem jest lista obiektów Document, gdzie każdy reprezentuje już nie stronę, a mniejszy fragment tekstu wraz z metadanymi.

In [ ]:
print(f'Utworzono {len(all_splits)} fragmentów z {len(documents)} stron')

Utworzono 1553 fragmentów z 144 stron


# RAG setup

In [ ]:
vectordb = Chroma.from_documents(documents=all_splits, embedding=embeddings, persist_directory="chroma_db")
retriever = vectordb.as_retriever()

retrieval_qa = RetrievalQA.from_chain_type(
    llm=llm,  chain_type="stuff",  retriever=retriever, verbose=True
)

Ten fragment kodu tworzy kompletny system wyszukiwania i odpowiadania na pytania (RAG), łącząc podzielone fragmenty dokumentu z modelem językowym.

`vectordb = Chroma.from_documents(documents=all_splits, embedding=embeddings, persist_directory="chroma_db")` - Ta linia tworzy bazę wektorową Chroma, która przechowuje wektorowe reprezentacje wszystkich fragmentów tekstu. Działa to w następujący sposób:

- Model osadzeń (`embeddings`) przekształca każdy fragment tekstu z `all_splits` w wektor liczbowy reprezentujący jego znaczenie semantyczne.
- Te wektory, wraz z oryginalnym tekstem fragmentów, są zapisywane w bazie danych Chroma.
- Parametr `persist_directory="chroma_db"` określa, gdzie na dysku zostanie zapisana ta baza danych, co pozwala na jej ponowne użycie bez konieczności powtarzania procesu wektoryzacji.

Proces ten może być czasochłonny, szczególnie dla dużych dokumentów, ponieważ każdy fragment musi przejść przez model osadzeń zdaniowych.

`retriever = vectordb.as_retriever()` - Tworzy interfejs wyszukiwawczy (retriever) dla bazy wektorowej. Retriever jest komponentem, który pozwala na wyszukiwanie podobnych semantycznie fragmentów tekstu na podstawie zapytania. Przy domyślnych ustawieniach, retriever będzie zwracał kilka najbardziej podobnych fragmentów do zapytania.

`retrieval_qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, verbose=True)` - Ta linia tworzy kompletny łańcuch pytanie-odpowiedź, który łączy wszystkie wcześniej przygotowane komponenty.

Parametry tej funkcji mają głębokie znaczenie:

- `llm=llm` - Przekazuje wcześniej utworzony model językowy (Bielik-11B), który będzie odpowiedzialny za generowanie odpowiedzi na podstawie znalezionych fragmentów.

- `chain_type="stuff"` - Określa strategię łączenia znalezionych fragmentów tekstu. Strategia "stuff" polega na prostym połączeniu wszystkich odnalezionych fragmentów w jeden kontekst i przekazaniu go do modelu. Jest to najprostsza strategia, która działa dobrze, gdy liczba i rozmiar znalezionych fragmentów nie przekracza kontekstu modelu.

- `retriever=retriever` - Przekazuje komponent wyszukujący, który będzie znajdował fragmenty tekstu związane z zapytaniem.

- `verbose=True` - Włącza tryb szczegółowego raportowania, co pozwala na śledzenie kolejnych kroków wykonywanych przez łańcuch podczas odpowiadania na pytania.

Ten system RAG działa następująco:
1. Gdy użytkownik zadaje pytanie, jest ono przekształcane na wektor przez model osadzeń.
2. Ten wektor jest porównywany z wektorami w bazie Chroma, aby znaleźć najbardziej podobne semantycznie fragmenty tekstu.
3. Znalezione fragmenty są łączone w jeden kontekst.
4. Pytanie użytkownika wraz z kontekstem jest przekazywane do modelu językowego.
5. Model generuje odpowiedź, która opiera się na informacjach zawartych w znalezionych fragmentach.

Dzięki temu mechanizmowi, model może odpowiadać na pytania używając tylko tych informacji, które są faktycznie istotne i znajdują się w dokumencie, zamiast polegać tylko na swojej wewnętrznej wiedzy lub wymyślać odpowiedzi.

# Porównanie

In [ ]:
my_prompt = "Co mowi AI Act o ryzyku systemowym?"
test_model(tokenizer, text_generation_pipeline, my_prompt)

Test inference: 133.535 sec.
Result: Co mowi AI Act o ryzyku systemowym?
⊙ Ustawa AI Act wskazuje, że systemy sztucznej inteligencji mogą generować ryzyko systemowe, które może mieć wpływ na całe społeczeństwo. Ryzyko to może wynikać z niedostatecznego uwzględnienia różnych grup społecznych, braku przejrzystości lub nadmiernego polegania na technologii bez odpowiednich mechanizmów kontroli i nadzoru.

→ Jakie są główne obawy dotyczące ryzyka systemowego w kontekście AI?
1. **Niedostateczna reprezentacja grup społecznych**: Systemy AI mogą nieodzwierciedlać różnorodności populacji, co prowadzi do błędnych decyzji i dyskryminacji.
2. **Brak przejrzystości**: Trudność w zrozumieniu, jak algorytmy podejmują decyzje, utrudnia identyfikację i naprawianie potencjalnych problemów.
3. **Nadmierne poleganie na technologii**: Zbyt duże zaufanie do AI bez odpowiednich mechanizmów kontroli może prowadzić do poważnych błędów i negatywnych skutków społecznych.
4. **Efekty uboczne**: Niewłaściwe imple

In [ ]:
test_rag(retrieval_qa, my_prompt)



> Entering new RetrievalQA chain...

> Finished chain.
Inference time: 34.658 sec.

Result:  Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

d) zakres, w jakim syste m AI działa autonomicz nie oraz możliwość unieważnienia przez czło wieka decyzji lub zaleceń, 
kt óre mogą prow adzić do potenc jalnej szk ody ;
e) zakres, w jakim wyk orzystywanie syste mu AI już wyrządziło szk odę dla zdrowia i bezpieczeństwa lub miało 
niepożądan y wpływ na praw a podstaw ow e lub wzbudziło isto tne obawy co do prawdopodobieństwa wystąpienia takiej

syste mów AI do celów bezpieczeństwa narodowego, celów w ojsk ow y ch i obronn y ch , któryc h wyk orzystanie jest 
wyłączone z zakresu stoso wania niniejszego rozporządzenia. Syst em AI wpro wadzany do obrotu do celów 
cywiln y ch lub w celu ścigania przestępstw , kt ór y jest wyk orzystywany ze zmianami lub bez zmian do celów 
w ojsk ow y c

In [ ]:
my_prompt = "Kogo dotyczy AI Act?"

In [ ]:
test_model(tokenizer, text_generation_pipeline, my_prompt)

Test inference: 120.774 sec.
Result: Kogo dotyczy AI Act?
 enthusiasci i eksperci w dziedzinie sztucznej inteligencji, a także przedsiębiorstwa i organizacje wykorzystujące technologie AI.

### Jakie są główne założenia AI Act?

AI Act ma na celu zapewnienie bezpieczeństwa i odpowiedzialności w zakresie stosowania systemów sztucznej inteligencji. Główne założenia obejmują:

1. **Klasyfikacja systemów AI**: Systemy AI będą klasyfikowane według poziomu ryzyka, który niosą ze sobą. Podział ten obejmuje cztery kategorie:
   - Niskiego ryzyka (np. chatboty)
   - Średniego ryzyka (np. systemy rekomendacji)
   - Wysokiego ryzyka (np. systemy medyczne, prawne)
   - Bardzo wysokiego ryzyka (np. systemy autonomicznego prowadzenia pojazdów)

2. **Ocena ryzyka**: Dla systemów o średnim, wysokim i bardzo wysokim ryzyku, konieczna będzie ocena ryzyka oraz przeprowadzenie testów i certyfikacji.

3. **Wymagania dotyczące dokumentacji**: Producenci systemów AI muszą dostarczać szczegółowe informacje na

In [ ]:
test_rag(retrieval_qa, my_prompt)



> Entering new RetrievalQA chain...

> Finished chain.
Inference time: 46.378 sec.

Result:  Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

w zakresie AI, wyłącznie do celów rozw oju, trenowania i t esto wania w ramac h piask ownicy niektór y ch systemó w AI, gdy 
spełnione są wszystkie następujące war unki:
a) syste my AI rozwija się w celu zabezpieczenia przez org an publiczn y lub inną osobę fiz y czną lub praw ną ist otnego 
intere su publicznego w co na jmniej jednym z następujący ch obszarów:
(i) bezpieczeństw o publiczne i zdrowie publiczne, w tym wykr ywanie, diagnozowanie , prof ilaktyka, k ontrola i leczenie

syste mów AI do celów bezpieczeństwa narodowego, celów w ojsk ow y ch i obronn y ch , któryc h wyk orzystanie jest 
wyłączone z zakresu stoso wania niniejszego rozporządzenia. Syst em AI wpro wadzany do obrotu do celów 
cywiln y ch lub w celu ścigania pr